In [ ]:
import pandas as pd
import numpy as np
import datetime



In [ ]:
df_aemet=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_AEMT", encoding="utf-8")
df_omie=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_OMIE", encoding="utf-8")
df_esios_pred=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS", encoding="utf-8")
df_esios_expl=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS_explicativo", encoding="utf-8")

In [ ]:
df_aemet.columns.tolist()
df_aemet.head()


In [ ]:
# Claves + las 6 variables que nos quedamos
columnas_interes = [
    'fecha',        # clave temporal
    'nombre',   # clave de estación (para el pivot)
    'tmed',         # temperatura media  -> demanda
    'tmax',         # temperatura máxima  -> picos de demanda (verano)
    'tmin',         # temperatura mínima  -> picos de demanda (invierno)ç
    'velmedia',     # viento medio        -> generación eólica
    'racha',        # racha máxima        -> eventos de viento
    'sol',          # horas de sol        -> generación solar
    'prec',         # precipitación       -> hidráulica / nubosidad
]

df_aemet = df_aemet[columnas_interes]



In [ ]:
df_aemet= df_aemet.pivot(index='fecha', columns='nombre', values=['tmed', 'tmax', 'velmedia', 'racha', 'sol', 'prec'])

In [ ]:
import unicodedata

def limpia(txt):
    txt = txt.lower().replace(',', '').replace(' ', '_')
    txt = ''.join(c for c in unicodedata.normalize('NFKD', txt)
                  if not unicodedata.combining(c))
    return txt

df_aemet.columns = [f"{var}_{limpia(est)}" for var, est in df_aemet.columns]
df_aemet = df_aemet.reset_index()

In [ ]:
df_aemet.columns.tolist() # ['fecha', 'tmed_alcazar_de_san_juan', ...]
df_aemet.shape              # (1277, 97)

In [ ]:
cobertura = (df_aemet.isna().mean() * 100).round(2).sort_values(ascending=False)

# 1) las 100% vacías (las que se tiran seguro)
print("Columnas al 100% NaN:")
print(cobertura[cobertura == 100])

# 2) las que tienen huecos parciales (0 < NaN < 100) -> se imputan luego
print("\nHuecos parciales:")
print(cobertura[(cobertura > 0) & (cobertura < 100)])

In [ ]:
muertas = df_aemet.columns[df_aemet.isna().mean() == 1.0].tolist()
df_aemet = df_aemet.drop(columns=muertas)

print(df_aemet.shape)          # 87 si no añadiste tmin; 103 si sí la añadiste
df_aemet.isna().mean().max()   # debe ser < 0.20 (ya no queda ningún 100%)

In [ ]:
df_aemet['fecha'] = pd.to_datetime(df_aemet['fecha'])
df_aemet['fecha'].dtype

In [ ]:
muertas = df_aemet.columns[df_aemet.isna().mean() == 1.0].tolist()
df_aemet = df_aemet.drop(columns=muertas)
df_aemet.shape   # esperado: (1277, 87)  -> 97 - 10

In [ ]:
df_omie.loc[(df_omie.ano==2023)&(df_omie.mes==10)&(df_omie.dia==29),'hora'].max()  # ¿25?
df_omie.loc[(df_omie.ano==2023)&(df_omie.mes==3)&(df_omie.dia==26),'hora'].max()    # ¿23?                         # ¿empieza en 1 (1-based) o en 0?

In [ ]:

# 1. día local a medianoche (desde ano/mes/dia)
dia = pd.to_datetime(df_omie[['ano','mes','dia']].rename(
        columns={'ano':'year','mes':'month','dia':'day'}))

# 2. ancla UTC (medianoche local -> UTC)
ancla = dia.dt.tz_localize('Europe/Madrid').dt.tz_convert('UTC')

# 3. paso por día: 60 min si el día tiene <=25 periodos, si no 15 min
n = df_omie.groupby(['ano','mes','dia'])['hora'].transform('size')
paso = np.where(n <= 25, 60, 15)

# 4. timestamp real = ancla + (hora-1)*paso, como tiempo transcurrido
df_omie['datetime_utc'] = ancla + pd.to_timedelta((df_omie['hora']-1)*paso, unit='m')


In [ ]:
df_omie.groupby(['ano','mes','dia']).size().value_counts()   # verás 24, 96 y algún 23/25/92/100
df_omie['datetime_utc'].duplicated().sum()                   # 0
df_omie.loc[0, 'datetime_utc']                               # 2022-12-31 23:00:00+00:00

In [ ]:
df_omie['datetime_utc'] = df_omie['datetime_utc'].dt.floor('h')

df_omie = (df_omie.groupby('datetime_utc')[['precio_espana','precio_portugal']]
                 .mean()
                 .reset_index())

In [ ]:
df_omie['datetime_utc'].duplicated().sum()     # 0 -> una fila por hora
df_omie.shape                              # ~ nº de horas de la serie
df_omie.head()
df_omie.tail()

In [ ]:
print(df_omie['datetime_utc'].min(), df_omie['datetime_utc'].max())
print(df_esios_pred['datetime_utc'].min(), df_esios_pred['datetime_utc'].max())
print(df_esios_expl['datetime_utc'].min(), df_esios_expl['datetime_utc'].max())


In [ ]:
df_esios_pred['datetime_utc'].duplicated().sum()   # 0
df_esios_expl['datetime_utc'].duplicated().sum()   # 0  (aplica el mismo fix a expl si no lo hiciste)

In [ ]:
df_esios_pred['datetime_utc'] = pd.to_datetime(df_esios_pred['datetime_utc'], utc=True)
df_esios_expl['datetime_utc'] = pd.to_datetime(df_esios_expl['datetime_utc'], utc=True)


In [ ]:
df_esios_pred['datetime_utc'].duplicated().sum()   # 
0
df_esios_expl['datetime_utc'].duplicated().sum()

In [ ]:
df_esios_expl.shape


In [ ]:
tabla_final = df_omie.merge(df_esios_pred, on='datetime_utc', how='left')
tabla_final = tabla_final.merge(df_esios_expl, on='datetime_utc', how='left')


In [ ]:
# Convertir 'fecha' a medianoche en zona horaria Europe/Madrid y hacer naive
tabla_final['fecha'] = (tabla_final['datetime_utc']
                                .dt.tz_convert('Europe/Madrid')
                                .dt.normalize()
                                .dt.tz_localize(None))

In [ ]:
print(len(df_omie), len(tabla_final))

In [ ]:
tabla_final['fecha'] = (tabla_final['datetime_utc']
                        .dt.tz_convert('Europe/Madrid')   # UTC -> Madrid
                        .dt.normalize()                   # a medianoche del día local
                        .dt.tz_localize(None))            # naive, para casar con AEMET

In [ ]:
tabla_final = tabla_final.merge(df_aemet, on='fecha', how='left')

In [ ]:

print(len(tabla_final))                    # sigue 30575 (AEMET es única por fecha, no infla)
tabla_final['tmed_madrid_aeropuerto'].notna().sum()   # >0 -> AEMET se pegó
tabla_final[['datetime_utc','fecha','tmed_madrid_aeropuerto']].head(30)   # ver el broadcast: 24 h con el mismo valor

In [ ]:
tabla_final.shape

In [ ]:
tabla_final.to_parquet('../data/interim/tabla_maestra_estructural.parquet', index=False)